# NRP-Exp: Bounded Preprocessing Validation

This notebook provides executable, bounded software-validation checks for the NRP-Exp preprocessing pipeline.

The notebook focuses on small controlled formulations for which the expected behaviour can be inspected directly. It covers:

1. Implication enforcement.
2. Coupling and static value-dependency absorption.
3. Multiplier interactions defined over a strict subset of a larger coupling component.
4. Exclusion branching.
5. Multi-subproblem optimisation.
6. Global Pareto-front recovery through dominance filtering.
7. Comparison against an independent bounded dominance oracle.

These checks are intended to complement the framework validation reported in the manuscript. They do not constitute a general proof of semantic equivalence for every theoretically possible interaction pattern

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Locate the project root containing the src directory."""

    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing "
        "the 'src' directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VALIDATION_CASES_DIR = (
    PROJECT_ROOT
    / "examples"
    / "instances"
    / "validation"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Validation cases: {VALIDATION_CASES_DIR}")

Project root: C:\Users\imagu\OneDrive\Escritorio\NRPExperimentFramework
Validation cases: C:\Users\imagu\OneDrive\Escritorio\NRPExperimentFramework\examples\instances\validation


## Software and Validation Artefacts

This notebook accompanies NRP-Exp version v1.2.0.

The controlled JSON cases used below are distributed in:

examples/instances/validation/

The notebook uses the same loader, preprocessing engine, evaluator, problem representation, and solver implementations as the main framework.

In [2]:
import json
from itertools import product

import pandas as pd

from src.application.use_cases.run_optimization import (
    combine_partial_pareto_fronts,
    run_optimization_models,
)
from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.core.solution import Solution
from src.domain.plugins.nrp import NRPPlugin
from src.solving.algorithms.brute_force import (
    BruteForceSolver,
)
from src.solving.core.evaluator import SolutionEvaluator
from src.solving.core.problem import OptimizationProblem

## Shared Helper Functions

The helper functions below load a controlled JSON case, execute the complete preprocessing pipeline, and display the resulting model variants.

All preprocessing checks use the actual pipeline. Processed models are not constructed manually.

In [3]:
def load_validation_case(file_name: str):
    """Load a validation case using the framework JSON loader."""

    case_path = VALIDATION_CASES_DIR / file_name

    if not case_path.exists():
        raise FileNotFoundError(
            f"Validation case not found: {case_path}"
        )

    raw_text = case_path.read_text(
        encoding="utf-8"
    )

    raw_data = json.loads(raw_text)

    model = GenericJSONLoader.load(
        json_input=raw_text,
        plugin=NRPPlugin,
    )

    return raw_data, model


def preprocess_model(base_model):
    """Execute the complete preprocessing pipeline."""

    processed_models = PreprocessingEngine.process(
        initial_model=base_model,
        pipeline_options={},
    )

    if not processed_models:
        raise RuntimeError(
            "The preprocessing pipeline produced no models."
        )

    return processed_models


def summarise_processed_models(processed_models):
    """Return a structural summary of processed variants."""

    rows = []

    for index, model in enumerate(processed_models):
        rows.append(
            {
                "variant_index": index,
                "branch_name": getattr(
                    model,
                    "branch_name",
                    f"Variant {index + 1}",
                ),
                "items": len(model.items),
                "precedences": len(
                    model.relationships.get(
                        "precedences",
                        [],
                    )
                ),
                "couplings": len(
                    model.relationships.get(
                        "couplings",
                        [],
                    )
                ),
                "exclusions": len(
                    model.relationships.get(
                        "exclusions",
                        [],
                    )
                ),
                "dynamic_value_dependencies": len(
                    model.relationships.get(
                        "value_dependencies",
                        [],
                    )
                ),
            }
        )

    return pd.DataFrame(rows)

1. Implication Enforcement

The preprocessing pipeline preserves implication relationships.

Operational enforcement of implications is performed by compatible solver implementations. The check below verifies that solutions returned by the solver satisfy the declared precedence semantics.

For the relationship:

r1 requires r2

selecting r1 therefore requires r2 to be selected as well. Consequently, the solver must never return a solution containing r1 without also containing r2.

In [4]:
from src.solving.algorithms.brute_force import (
    BruteForceSolver
)
from src.solving.core.problem import (
    OptimizationProblem
)

_, implication_model = load_validation_case(
    "implications.json"
)

implication_variants = preprocess_model(
    implication_model
)

assert len(implication_variants) == 1

implication_processed = implication_variants[0]

problem = OptimizationProblem(
    model=implication_processed,
    objectives={
        "value": "max",
    },
    constraints={},
)

solver = BruteForceSolver(
    max_evaluations=100
)

front = solver.solve(problem)

rows = []

for solution in front.solutions:

    selected = set(solution.selected_ids)

    rows.append(
        {
            "solution_id": solution.id,
            "selected": sorted(selected),
        }
    )

    if "r1" in selected:
        assert "r2" in selected

display(pd.DataFrame(rows))

print(
    "Implication enforcement validated."
)

,solution_id,selected
0,BF_1,"[r1, r2, r3]"


Implication enforcement validated.


2. Coupling and Value-Based Interaction Processing

The following case contains two coupled requirements:

r1 = 10
r2 = 20

A multiplier of 2.0 applies when both requirements are selected.

Before preprocessing:

(10 + 20) × 2 = 60

After coupling, the two requirements become one decision item. Static absorption must preserve the same value.

In [5]:
_, coupling_model = load_validation_case(
    "coupling_multiplier.json"
)

value_before = SolutionEvaluator(
    coupling_model
).evaluate_attribute(
    {"r1", "r2"},
    "value",
)

coupling_variants = preprocess_model(
    coupling_model
)

assert len(coupling_variants) == 1

coupling_processed = coupling_variants[0]

assert len(coupling_processed.items) == 1

coupled_item_id = next(
    iter(coupling_processed.items)
)

value_after = SolutionEvaluator(
    coupling_processed
).evaluate_attribute(
    {coupled_item_id},
    "value",
)

coupling_result = pd.Series(
    {
        "Before preprocessing": value_before,
        "After preprocessing": value_after,
        "Equivalent": value_before == value_after,
    },
    name="Coupling and multiplier",
)

display(coupling_result.to_frame())

assert value_before == 60.0
assert value_after == 60.0
assert value_before == value_after

print("Static multiplier absorption check passed.")

,Coupling and multiplier
Before preprocessing,60.0
After preprocessing,60.0
Equivalent,True


Static multiplier absorption check passed.


3. Reviewer Multiplier Edge Case

This case reproduces the interaction pattern identified during review.

Requirements:

r1 = 10
r2 = 20
r3 = 100
Coupling:

r1 -- r2
r2 -- r3

Multiplier:

(r1,r2) × 2

Expected value:

(10 + 20) × 2 + 100 = 160

The preprocessing transformation must preserve the contribution of the original activating subset.

In [6]:
_, edge_model = load_validation_case(
    "reviewer_multiplier_edge.json"
)

edge_before = SolutionEvaluator(
    edge_model
).evaluate_attribute(
    {"r1", "r2", "r3"},
    "value",
)

edge_variants = preprocess_model(
    edge_model
)

assert len(edge_variants) == 1

edge_processed = edge_variants[0]

assert len(edge_processed.items) == 1

edge_item_id = next(
    iter(edge_processed.items)
)

edge_after = SolutionEvaluator(
    edge_processed
).evaluate_attribute(
    {edge_item_id},
    "value",
)

edge_result = pd.Series(
    {
        "Expected": 160.0,
        "Before preprocessing": edge_before,
        "After preprocessing": edge_after,
        "Equivalent": edge_before == edge_after,
    },
    name="Reviewer multiplier edge case",
)

display(edge_result.to_frame())

assert edge_before == 160.0
assert edge_after == 160.0
assert edge_before == edge_after

print("Reviewer multiplier edge-case check passed.")

,Reviewer multiplier edge case
Expected,160.0
Before preprocessing,160.0
After preprocessing,160.0
Equivalent,True


Reviewer multiplier edge-case check passed.


4. Exclusion Branching
The controlled exclusion case contains two independent mutual exclusions:

- r1 XOR r2
- r3 XOR r4

Two independent binary exclusion choices must generate:

2^2 = 4

optimisation subproblems.

Each resulting subproblem must retain exactly one requirement from each exclusion pair.

In [7]:
_, exclusion_model = load_validation_case(
    "exclusion_global_front.json"
)

exclusion_variants = preprocess_model(
    exclusion_model
)

display(
    summarise_processed_models(
        exclusion_variants
    )
)

assert len(exclusion_variants) == 4

exclusion_pairs = [
    {"r1", "r2"},
    {"r3", "r4"},
]

branch_rows = []

for model in exclusion_variants:
    surviving_ids = set(model.items)

    for excluded_pair in exclusion_pairs:
        assert len(
            surviving_ids & excluded_pair
        ) == 1

    branch_rows.append(
        {
            "branch": getattr(
                model,
                "branch_name",
                "Unlabelled branch",
            ),
            "surviving_items": sorted(
                surviving_ids
            ),
            "remaining_exclusions": len(
                model.relationships.get(
                    "exclusions",
                    [],
                )
            ),
        }
    )

branch_df = pd.DataFrame(branch_rows)

display(branch_df)

assert (
    branch_df["remaining_exclusions"] == 0
).all()

print("Exclusion branching check passed.")

,variant_index,branch_name,items,precedences,couplings,exclusions,dynamic_value_dependencies
0,0,Excl(r1) ➔ Excl(r3),2,0,0,0,0
1,1,Excl(r1) ➔ Excl(r4),2,0,0,0,0
2,2,Excl(r2) ➔ Excl(r3),2,0,0,0,0
3,3,Excl(r2) ➔ Excl(r4),2,0,0,0,0


,branch,surviving_items,remaining_exclusions
0,Excl(r1) ➔ Excl(r3),"[r2, r4]",0
1,Excl(r1) ➔ Excl(r4),"[r2, r3]",0
2,Excl(r2) ➔ Excl(r3),"[r1, r4]",0
3,Excl(r2) ➔ Excl(r4),"[r1, r3]",0


Exclusion branching check passed.


5. Multi-Subproblem Optimisation

Each exclusion-generated model is formulated as an independent optimisation problem.

The same exhaustive solver configuration is applied to every subproblem. The resulting partial Pareto fronts are then combined and filtered using global dominance.

This operation corresponds to recovering the non-dominated solution set of the complete exclusion-constrained formulation.

In [8]:
def brute_force_factory():
    """Create a fresh solver instance for each subproblem."""

    return BruteForceSolver(
        max_evaluations=10000
    )


multi_result = run_optimization_models(
    models=exclusion_variants,
    objectives={
        "value": "max",
        "cost": "min",
    },
    constraints={},
    solver_factory=brute_force_factory,
)

partial_summary = []

for index, partial_result in enumerate(
    multi_result.partial_results
):
    partial_summary.append(
        {
            "subproblem_index": index,
            "branch": getattr(
                exclusion_variants[index],
                "branch_name",
                f"Subproblem {index + 1}",
            ),
            "partial_front_size": len(
                partial_result
                .pareto_front
                .solutions
            ),
        }
    )

display(pd.DataFrame(partial_summary))

print(
    "Global Pareto-front size:",
    len(multi_result.pareto_front.solutions),
)

display(multi_result.pareto_df)

assert len(multi_result.partial_results) == 4
assert all(
    solution.is_feasible
    for solution
    in multi_result.pareto_front.solutions
)
assert not multi_result.pareto_df.empty

,subproblem_index,branch,partial_front_size
0,0,Excl(r1) ➔ Excl(r3),4
1,1,Excl(r1) ➔ Excl(r4),4
2,2,Excl(r2) ➔ Excl(r3),4
3,3,Excl(r2) ➔ Excl(r4),3


Global Pareto-front size: 7


,id,selected_ids,selected_ids_str,is_feasible,value,cost
0,SP1::Excl(r1) ➔ Excl(r3)::BF_1,"[r2, r4]","r2, r4",True,35.0,23.0
1,SP1::Excl(r1) ➔ Excl(r3)::BF_2,[r2],r2,True,15.0,8.0
2,SP1::Excl(r1) ➔ Excl(r3)::BF_4,[],,True,0.0,0.0
3,SP2::Excl(r1) ➔ Excl(r4)::BF_1,"[r2, r3]","r2, r3",True,27.0,12.0
4,SP2::Excl(r1) ➔ Excl(r4)::BF_3,[r3],r3,True,12.0,4.0
5,SP3::Excl(r2) ➔ Excl(r3)::BF_1,"[r1, r4]","r1, r4",True,30.0,20.0
6,SP4::Excl(r2) ➔ Excl(r4)::BF_1,"[r1, r3]","r1, r3",True,22.0,9.0


6. Independent Dominance Oracle

The next check calculates the expected global front using an independent, bounded dominance implementation.

The oracle deliberately does not call:

Solution.dominates();
ParetoFront.filter_non_dominated();
combine_partial_pareto_fronts().

This provides an independent comparison for the controlled, finite solution set returned by the four subproblems.

In [9]:
OBJECTIVE_DIRECTIONS = {
    "value": "max",
    "cost": "min",
}


def independently_dominates(
    first: Solution,
    second: Solution,
    directions: dict,
) -> bool:
    """Independent Pareto-dominance implementation."""

    if first.is_feasible and not second.is_feasible:
        return True

    if not first.is_feasible:
        return False

    if not second.is_feasible:
        return False

    no_worse_in_all = True
    strictly_better_in_one = False

    for objective_name, direction in directions.items():
        first_value = float(
            first.objectives[objective_name]
        )
        second_value = float(
            second.objectives[objective_name]
        )

        if direction == "max":
            if first_value < second_value:
                no_worse_in_all = False
                break

            if first_value > second_value:
                strictly_better_in_one = True

        elif direction == "min":
            if first_value > second_value:
                no_worse_in_all = False
                break

            if first_value < second_value:
                strictly_better_in_one = True

        else:
            raise ValueError(
                f"Unsupported objective direction: "
                f"{direction!r}"
            )

    return (
        no_worse_in_all
        and strictly_better_in_one
    )


def independent_non_dominated_oracle(
    solutions,
    directions,
):
    """Return non-dominated feasible solutions."""

    feasible_solutions = [
        solution
        for solution in solutions
        if solution.is_feasible
    ]

    non_dominated = []

    for candidate_index, candidate in enumerate(
        feasible_solutions
    ):
        dominated = False

        for other_index, other in enumerate(
            feasible_solutions
        ):
            if candidate_index == other_index:
                continue

            if independently_dominates(
                first=other,
                second=candidate,
                directions=directions,
            ):
                dominated = True
                break

        if not dominated:
            non_dominated.append(candidate)

    return non_dominated

In [10]:
all_partial_solutions = [
    solution
    for partial_result in multi_result.partial_results
    for solution
    in partial_result.pareto_front.solutions
]

oracle_front = independent_non_dominated_oracle(
    solutions=all_partial_solutions,
    directions=OBJECTIVE_DIRECTIONS,
)


def objective_vector(solution):
    return tuple(
        float(solution.objectives[name])
        for name in OBJECTIVE_DIRECTIONS
    )


framework_vectors = {
    objective_vector(solution)
    for solution
    in multi_result.pareto_front.solutions
}

oracle_vectors = {
    objective_vector(solution)
    for solution in oracle_front
}

comparison = pd.Series(
    {
        "Framework front size": len(
            framework_vectors
        ),
        "Oracle front size": len(
            oracle_vectors
        ),
        "Equivalent vectors": (
            framework_vectors
            == oracle_vectors
        ),
    },
    name="Global-front comparison",
)

display(comparison.to_frame())

print(
    f"Framework vectors: {len(framework_vectors)}"
)
print(
    f"Oracle vectors: {len(oracle_vectors)}"
)

print(
    "The global front matches the "
    "independent dominance oracle."
)

,Global-front comparison
Framework front size,7
Oracle front size,7
Equivalent vectors,True


Framework vectors: 7
Oracle vectors: 7
The global front matches the independent dominance oracle.


7. Four-Exclusion Boundary Check

The preprocessing engine can process more than three exclusions, even though the interactive All option is limited to at most three exclusions as an operational safeguard.

The controlled case contains four independent exclusions:

- r1 XOR r2
- r3 XOR r4
- r5 XOR r6
- r7 XOR r8

The preprocessing engine must generate:

2^4 = 16   subproblems.

This check distinguishes backend semantic capability from the user-interface execution limit. The three-exclusion limit applies only to the optional automatic execution of all exclusion-generated subproblems in the user interface and does not restrict the preprocessing pipeline itself.

In [11]:
_, four_exclusion_model = load_validation_case(
    "four_exclusions.json"
)

four_exclusion_variants = preprocess_model(
    four_exclusion_model
)

print(
    "Generated subproblems:",
    len(four_exclusion_variants),
)

assert len(four_exclusion_variants) == 16

four_pairs = [
    {"r1", "r2"},
    {"r3", "r4"},
    {"r5", "r6"},
    {"r7", "r8"},
]

branch_signatures = set()

for model in four_exclusion_variants:
    surviving_ids = set(model.items)

    signature = []

    for excluded_pair in four_pairs:
        surviving_pair = (
            surviving_ids
            & excluded_pair
        )

        assert len(surviving_pair) == 1

        signature.append(
            next(iter(surviving_pair))
        )

    branch_signatures.add(
        tuple(signature)
    )

display(
    summarise_processed_models(
        four_exclusion_variants
    )
)

print(
    "All sixteen exclusion combinations "
    "were generated exactly once."
)

Generated subproblems: 16


,variant_index,branch_name,items,precedences,couplings,exclusions,dynamic_value_dependencies
0,0,Excl(r1) ➔ Excl(r3) ➔ Excl(r5) ➔ Excl(r7),4,0,0,0,0
1,1,Excl(r1) ➔ Excl(r3) ➔ Excl(r5) ➔ Excl(r8),4,0,0,0,0
2,2,Excl(r1) ➔ Excl(r3) ➔ Excl(r6) ➔ Excl(r7),4,0,0,0,0
3,3,Excl(r1) ➔ Excl(r3) ➔ Excl(r6) ➔ Excl(r8),4,0,0,0,0
4,4,Excl(r1) ➔ Excl(r4) ➔ Excl(r5) ➔ Excl(r7),4,0,0,0,0
5,5,Excl(r1) ➔ Excl(r4) ➔ Excl(r5) ➔ Excl(r8),4,0,0,0,0
6,6,Excl(r1) ➔ Excl(r4) ➔ Excl(r6) ➔ Excl(r7),4,0,0,0,0
7,7,Excl(r1) ➔ Excl(r4) ➔ Excl(r6) ➔ Excl(r8),4,0,0,0,0
8,8,Excl(r2) ➔ Excl(r3) ➔ Excl(r5) ➔ Excl(r7),4,0,0,0,0
9,9,Excl(r2) ➔ Excl(r3) ➔ Excl(r5) ➔ Excl(r8),4,0,0,0,0


All sixteen exclusion combinations were generated exactly once.


In [12]:
assert value_before == value_after == 60.0

assert edge_before == edge_after == 160.0

assert len(exclusion_variants) == 4

assert (
    framework_vectors
    == oracle_vectors
)

assert len(four_exclusion_variants) == 16

print(
    "All bounded preprocessing validation "
    "checks passed successfully."
)

All bounded preprocessing validation checks passed successfully.


## Validation Summary

The bounded checks performed in this notebook provide evidence that:

- implication relationships are handled consistently within the supported optimisation workflow;
- coupling resolution preserves the evaluated multiplier semantics;
- the multiplier edge case identified during review is preserved before and after preprocessing;
- independent exclusions generate the expected subproblems;
- all generated subproblems can be optimised through the common solver contract;
- partial fronts can be combined through global dominance filtering;
- the resulting global front matches an independent bounded dominance oracle;
- four exclusions generate the expected sixteen subproblems.

These results provide bounded software-validation evidence for the controlled formulations considered here. They demonstrate semantic consistency for the evaluated interaction patterns, but do not constitute a formal proof of equivalence for all possible combinations of NRP interactions.

In [13]:
assert value_before == value_after == 60.0

assert edge_before == edge_after == 160.0

assert len(exclusion_variants) == 4

assert (
    framework_vectors
    == oracle_vectors
)

assert len(four_exclusion_variants) == 16

print(
    "All bounded preprocessing validation "
    "checks passed successfully."
)

All bounded preprocessing validation checks passed successfully.
